In [ ]:
# Basic hashing operations and hash variants
import mmh3
import struct
from typing import List, Union, Any

print("=== MMH3 Basic Hashing Operations ===")

def demonstrate_hash_variants():
    """Demonstrate different MurmurHash3 variants"""
    test_data = [
        "hello world",
        "data structure",
        "bloom filter",
        b"binary data",
        12345,
        3.14159
    ]
    
    results = []
    
    for data in test_data:
        # Convert to bytes if needed
        if isinstance(data, str):
            data_bytes = data.encode('utf-8')
            data_str = data
        elif isinstance(data, (int, float)):
            data_str = str(data)
            data_bytes = data_str.encode('utf-8')
        else:
            data_bytes = data
            data_str = str(data)
        
        # 32-bit hash
        hash32 = mmh3.hash(data_bytes)
        
        # 128-bit hash (returns one unsigned 128-bit integer)
        hash128 = mmh3.hash128(data_bytes)
        
        # 32-bit hash with seed
        hash32_seeded = mmh3.hash(data_bytes, seed=42)
        
        # 128-bit hash with seed  
        hash128_seeded = mmh3.hash128(data_bytes, seed=42)
        
        result = {
            'input': data_str,
            'hash32': hash32,
            'hash32_hex': f"0x{hash32 & 0xffffffff:08x}",
            'hash128': hash128,
            'hash32_seeded': hash32_seeded,
            'hash128_seeded': hash128_seeded
        }
        results.append(result)
    
    return results

# Demonstrate hash variants
hash_results = demonstrate_hash_variants()

print("Hash Results Comparison:")
for result in hash_results[:4]:  # Show first 4 for brevity
    print(f"Input: {result['input']}")
    print(f"  32-bit: {result['hash32']} ({result['hash32_hex']})")
    print(f"  128-bit: {result['hash128']}")
    print(f"  32-bit (seed=42): {result['hash32_seeded']}")
    print()

# Reference values from the mmh3 documentation
assert mmh3.hash(b"foo") == -156908512 and mmh3.hash(b"foo", seed=42) == -1322301282
assert mmh3.hash128(b"foo", seed=42) == 215966891540331383248189432718888555506
# The six distinct inputs hash to six distinct values, and every 128-bit hash fits in 128 bits
assert len({r['hash32'] for r in hash_results}) == len(hash_results)
assert all(0 <= r['hash128'] < 2**128 for r in hash_results)

# Demonstrate seed consistency
test_string = "consistent hashing test"
seeds = [0, 1, 42, 100, 999]

print("Seed Consistency Test:")
for seed in seeds:
    hash_value = mmh3.hash(test_string.encode('utf-8'), seed=seed)
    print(f"Seed {seed:3d}: {hash_value:11d} (0x{hash_value & 0xffffffff:08x})")
assert len({mmh3.hash(test_string.encode('utf-8'), seed=seed) for seed in seeds}) == len(seeds)

# Demonstrate deterministic behavior
print(f"\nDeterministic Test (same input, multiple calls):")
test_input = "deterministic test"
hashes = [mmh3.hash(test_input.encode('utf-8')) for _ in range(5)]
print(f"All hashes identical: {len(set(hashes)) == 1}")
print(f"Hash value: {hashes[0]}")
assert len(set(hashes)) == 1

print("Basic hashing operations completed successfully!")

In [ ]:
# Hash distribution analysis and collision detection
import collections
from typing import Dict, Set

print("=== MMH3 Hash Distribution Analysis ===")

class HashDistributionAnalyzer:
    def __init__(self, hash_function=None):
        self.hash_function = hash_function or (lambda x: mmh3.hash(x.encode('utf-8')))
    
    def analyze_distribution(self, data_set: List[str], bucket_count: int = 100) -> Dict:
        """Analyze hash distribution across buckets"""
        hash_values = [self.hash_function(item) for item in data_set]
        
        # Map to buckets
        bucket_assignments = [abs(hash_val) % bucket_count for hash_val in hash_values]
        bucket_counts = collections.Counter(bucket_assignments)
        
        # Distribution statistics
        filled_buckets = len(bucket_counts)
        empty_buckets = bucket_count - filled_buckets
        max_bucket_size = max(bucket_counts.values()) if bucket_counts else 0
        min_bucket_size = min(bucket_counts.values()) if bucket_counts else 0
        avg_bucket_size = len(data_set) / bucket_count
        
        # Calculate uniformity (chi-square test approximation)
        expected_count = len(data_set) / bucket_count
        chi_square = sum((count - expected_count) ** 2 / expected_count 
                        for count in bucket_counts.values())
        
        return {
            'total_items': len(data_set),
            'bucket_count': bucket_count,
            'filled_buckets': filled_buckets,
            'empty_buckets': empty_buckets,
            'load_factor': len(data_set) / bucket_count,
            'max_bucket_size': max_bucket_size,
            'min_bucket_size': min_bucket_size,
            'avg_bucket_size': avg_bucket_size,
            'chi_square': chi_square,
            'hash_values': hash_values[:10],  # Sample hash values
            'bucket_distribution': dict(list(bucket_counts.most_common(10)))
        }
    
    def detect_collisions(self, data_set: List[str]) -> Dict:
        """Detect hash collisions in dataset"""
        hash_to_data = collections.defaultdict(list)
        
        for item in data_set:
            hash_val = self.hash_function(item)
            hash_to_data[hash_val].append(item)
        
        collisions = {hash_val: items for hash_val, items in hash_to_data.items() 
                     if len(items) > 1}
        
        return {
            'total_items': len(data_set),
            'unique_hashes': len(hash_to_data),
            'collision_count': len(collisions),
            'collision_rate': len(collisions) / len(hash_to_data) if hash_to_data else 0,
            'max_collision_size': max(len(items) for items in collisions.values()) if collisions else 0,
            'sample_collisions': dict(list(collisions.items())[:3])
        }

# Generate test dataset with various patterns
def generate_test_data() -> List[str]:
    """Generate test data with different patterns"""
    data = []
    
    # Sequential data
    data.extend([f"user_{i:05d}" for i in range(1000)])
    
    # Dictionary words (simulated)
    words = ["apple", "banana", "cherry", "date", "elderberry", "fig", "grape"]
    data.extend([f"{word}_{i}" for word in words for i in range(100)])
    
    # Random-like data
    import random
    random.seed(42)
    data.extend([''.join(random.choices('abcdefghijklmnopqrstuvwxyz0123456789', k=8)) 
                for _ in range(500)])
    
    # Similar strings (test collision resistance)
    base = "similar_string"
    data.extend([f"{base}_{i}" for i in range(100)])
    
    return data

# Analyze hash distribution
analyzer = HashDistributionAnalyzer()
test_data = generate_test_data()

# Distribution analysis with different bucket sizes
bucket_sizes = [50, 100, 200, 500]
print("Distribution Analysis Results:")

for bucket_size in bucket_sizes:
    dist_analysis = analyzer.analyze_distribution(test_data, bucket_size)
    print(f"\nBucket size: {bucket_size}")
    print(f"  Load factor: {dist_analysis['load_factor']:.2f}")
    print(f"  Filled buckets: {dist_analysis['filled_buckets']}/{bucket_size}")
    print(f"  Max bucket size: {dist_analysis['max_bucket_size']}")
    print(f"  Chi-square: {dist_analysis['chi_square']:.2f}")
    assert dist_analysis['filled_buckets'] > 0.9 * bucket_size, dist_analysis['filled_buckets']

# Collision detection
collision_analysis = analyzer.detect_collisions(test_data)
print(f"\nCollision Analysis:")
print(f"Total items: {collision_analysis['total_items']}")
print(f"Unique hashes: {collision_analysis['unique_hashes']}")
print(f"Collisions found: {collision_analysis['collision_count']}")
print(f"Collision rate: {collision_analysis['collision_rate']*100:.4f}%")
assert collision_analysis['total_items'] == len(test_data)
assert collision_analysis['collision_rate'] < 0.001, collision_analysis['collision_count']

if collision_analysis['sample_collisions']:
    print("Sample collisions:")
    for hash_val, items in list(collision_analysis['sample_collisions'].items())[:2]:
        print(f"  Hash {hash_val}: {items[:3]}...")

print("Hash distribution analysis completed successfully!")

In [ ]:
# Bloom filter implementation using MurmurHash3
import math
from typing import Set, List, Tuple

print("=== MMH3 Bloom Filter Implementation ===")

class MMH3BloomFilter:
    def __init__(self, expected_items: int, false_positive_rate: float = 0.01):
        """
        Initialize Bloom filter with MMH3 hash functions
        
        Args:
            expected_items: Expected number of items to be added
            false_positive_rate: Desired false positive rate (0.01 = 1%)
        """
        self.expected_items = expected_items
        self.false_positive_rate = false_positive_rate
        
        # Calculate optimal bit array size
        self.bit_array_size = int(-(expected_items * math.log(false_positive_rate)) / (math.log(2) ** 2))
        
        # Calculate optimal number of hash functions
        self.num_hash_functions = int((self.bit_array_size / expected_items) * math.log(2))
        
        # Initialize bit array
        self.bit_array = [False] * self.bit_array_size
        
        # Track items added and statistics
        self.items_added = 0
        self.queries = 0
        self.false_positives = 0
    
    def _get_hash_values(self, item: str) -> List[int]:
        """Generate multiple hash values using different seeds"""
        hash_values = []
        for i in range(self.num_hash_functions):
            # Use different seeds for each hash function
            hash_val = mmh3.hash(item.encode('utf-8'), seed=i)
            # Map to bit array index
            index = abs(hash_val) % self.bit_array_size
            hash_values.append(index)
        return hash_values
    
    def add(self, item: str):
        """Add item to bloom filter"""
        hash_indices = self._get_hash_values(item)
        for index in hash_indices:
            self.bit_array[index] = True
        self.items_added += 1
    
    def contains(self, item: str) -> bool:
        """Check if item might be in the set (may have false positives)"""
        self.queries += 1
        hash_indices = self._get_hash_values(item)
        return all(self.bit_array[index] for index in hash_indices)
    
    def estimated_false_positive_rate(self) -> float:
        """Calculate current estimated false positive rate"""
        if self.items_added == 0:
            return 0.0
        
        # Probability that a bit is still 0
        p = (1 - 1/self.bit_array_size) ** (self.num_hash_functions * self.items_added)
        
        # False positive probability
        return (1 - p) ** self.num_hash_functions
    
    def get_statistics(self) -> Dict:
        """Get bloom filter statistics"""
        bits_set = sum(self.bit_array)
        fill_ratio = bits_set / self.bit_array_size
        
        return {
            'bit_array_size': self.bit_array_size,
            'num_hash_functions': self.num_hash_functions,
            'items_added': self.items_added,
            'queries': self.queries,
            'bits_set': bits_set,
            'fill_ratio': fill_ratio,
            'target_fp_rate': self.false_positive_rate,
            'estimated_fp_rate': self.estimated_false_positive_rate()
        }

def test_bloom_filter():
    """Test bloom filter with MMH3 hashes"""
    # Create bloom filter for 1000 items with 1% false positive rate
    bf = MMH3BloomFilter(expected_items=1000, false_positive_rate=0.01)
    
    # Add test data
    test_items = [f"item_{i:05d}" for i in range(800)]
    for item in test_items:
        bf.add(item)
    
    # Test membership for added items (should all be True)
    true_positives = 0
    for item in test_items[:100]:  # Test subset
        if bf.contains(item):
            true_positives += 1
    
    # Test with items NOT in the set (may have false positives)
    false_items = [f"false_item_{i:05d}" for i in range(200)]
    false_positives = 0
    for item in false_items:
        if bf.contains(item):
            false_positives += 1
    
    actual_fp_rate = false_positives / len(false_items)
    
    return bf, {
        'true_positives': true_positives,
        'true_positive_rate': true_positives / 100,
        'false_positives': false_positives,
        'actual_fp_rate': actual_fp_rate,
        'test_items_count': len(false_items)
    }

# Test bloom filter implementation
bloom_filter, test_results = test_bloom_filter()
bf_stats = bloom_filter.get_statistics()

print("Bloom Filter Configuration:")
print(f"Expected items: {bloom_filter.expected_items}")
print(f"Bit array size: {bf_stats['bit_array_size']:,}")
print(f"Hash functions: {bf_stats['num_hash_functions']}")
print(f"Target false positive rate: {bf_stats['target_fp_rate']*100:.2f}%")

print(f"\nBloom Filter Performance:")
print(f"Items added: {bf_stats['items_added']}")
print(f"Bits set: {bf_stats['bits_set']:,} ({bf_stats['fill_ratio']*100:.1f}%)")
print(f"Estimated FP rate: {bf_stats['estimated_fp_rate']*100:.2f}%")

print(f"\nTest Results:")
print(f"True positive rate: {test_results['true_positive_rate']*100:.1f}%")
print(f"False positives: {test_results['false_positives']}/{test_results['test_items_count']}")
print(f"Actual FP rate: {test_results['actual_fp_rate']*100:.2f}%")
# A Bloom filter never reports an added item as missing; false positives stay near the 1% target
assert test_results['true_positive_rate'] == 1.0
assert test_results['actual_fp_rate'] < 0.05, test_results['actual_fp_rate']

# Demonstrate different bloom filter sizes
print(f"\nBloom Filter Size Comparison:")
sizes = [(100, 0.1), (1000, 0.05), (5000, 0.01), (10000, 0.001)]

for items, fp_rate in sizes:
    temp_bf = MMH3BloomFilter(items, fp_rate)
    print(f"Items: {items:5d}, FP: {fp_rate*100:5.1f}% -> Bits: {temp_bf.bit_array_size:6d}, Functions: {temp_bf.num_hash_functions}")

print("Bloom filter implementation completed successfully!")

In [ ]:
# Consistent hashing and data partitioning for distributed systems
from typing import Dict, List, Tuple, Optional

print("=== MMH3 Consistent Hashing Implementation ===")

class ConsistentHashRing:
    def __init__(self, virtual_nodes_per_server: int = 150):
        """
        Consistent hash ring implementation using MMH3
        
        Args:
            virtual_nodes_per_server: Number of virtual nodes per server for better distribution
        """
        self.virtual_nodes_per_server = virtual_nodes_per_server
        self.ring = {}  # hash_value -> server_id
        self.servers = set()
        self.hash_ring_sorted = []  # Sorted list of hash values
    
    def _hash_server(self, server_id: str, virtual_node_id: int) -> int:
        """Generate hash for server virtual node"""
        key = f"{server_id}:{virtual_node_id}"
        return mmh3.hash(key.encode('utf-8'))
    
    def _hash_key(self, key: str) -> int:
        """Generate hash for data key"""
        return mmh3.hash(key.encode('utf-8'))
    
    def add_server(self, server_id: str):
        """Add server to the hash ring"""
        if server_id in self.servers:
            return
        
        self.servers.add(server_id)
        
        # Add virtual nodes for this server
        for i in range(self.virtual_nodes_per_server):
            hash_val = self._hash_server(server_id, i)
            self.ring[hash_val] = server_id
        
        # Update sorted ring
        self.hash_ring_sorted = sorted(self.ring.keys())
    
    def remove_server(self, server_id: str):
        """Remove server from the hash ring"""
        if server_id not in self.servers:
            return
        
        self.servers.remove(server_id)
        
        # Remove virtual nodes for this server
        keys_to_remove = [hash_val for hash_val, sid in self.ring.items() if sid == server_id]
        for hash_val in keys_to_remove:
            del self.ring[hash_val]
        
        # Update sorted ring
        self.hash_ring_sorted = sorted(self.ring.keys())
    
    def get_server(self, key: str) -> Optional[str]:
        """Get the server responsible for a given key"""
        if not self.hash_ring_sorted:
            return None
        
        key_hash = self._hash_key(key)
        
        # Find the first server with hash >= key_hash
        for ring_hash in self.hash_ring_sorted:
            if ring_hash >= key_hash:
                return self.ring[ring_hash]
        
        # Wrap around to the first server
        return self.ring[self.hash_ring_sorted[0]]
    
    def get_servers_for_replication(self, key: str, replication_factor: int = 3) -> List[str]:
        """Get multiple servers for replication"""
        if not self.hash_ring_sorted or replication_factor <= 0:
            return []
        
        key_hash = self._hash_key(key)
        servers = []
        seen_servers = set()
        
        # Find starting position
        start_idx = 0
        for i, ring_hash in enumerate(self.hash_ring_sorted):
            if ring_hash >= key_hash:
                start_idx = i
                break
        
        # Collect unique servers for replication
        idx = start_idx
        while len(servers) < replication_factor and len(servers) < len(self.servers):
            server_id = self.ring[self.hash_ring_sorted[idx]]
            if server_id not in seen_servers:
                servers.append(server_id)
                seen_servers.add(server_id)
            idx = (idx + 1) % len(self.hash_ring_sorted)
        
        return servers
    
    def analyze_distribution(self, test_keys: List[str]) -> Dict:
        """Analyze key distribution across servers"""
        if not self.servers:
            return {}
        
        server_counts = {server: 0 for server in self.servers}
        key_assignments = {}
        
        for key in test_keys:
            server = self.get_server(key)
            if server:
                server_counts[server] += 1
                key_assignments[key] = server
        
        # Calculate distribution statistics
        total_keys = len(test_keys)
        expected_per_server = total_keys / len(self.servers)
        
        distribution_stats = {
            'total_keys': total_keys,
            'server_count': len(self.servers),
            'expected_per_server': expected_per_server,
            'server_loads': server_counts,
            'min_load': min(server_counts.values()) if server_counts else 0,
            'max_load': max(server_counts.values()) if server_counts else 0,
            'load_balance_ratio': (min(server_counts.values()) / max(server_counts.values()) 
                                 if server_counts and max(server_counts.values()) > 0 else 0)
        }
        
        return distribution_stats

def test_consistent_hashing():
    """Test consistent hashing implementation"""
    # Initialize hash ring
    hash_ring = ConsistentHashRing(virtual_nodes_per_server=100)
    
    # Add servers
    servers = ['server-001', 'server-002', 'server-003', 'server-004', 'server-005']
    for server in servers:
        hash_ring.add_server(server)
    
    # Generate test keys
    test_keys = [f"data_key_{i:06d}" for i in range(10000)]
    
    # Analyze initial distribution
    initial_distribution = hash_ring.analyze_distribution(test_keys)
    
    # Test replication
    sample_key = "important_data_key"
    replicas = hash_ring.get_servers_for_replication(sample_key, replication_factor=3)
    
    # Test server removal impact
    removed_server = 'server-003'
    affected_keys = []
    
    for key in test_keys[:1000]:  # Test subset
        original_server = hash_ring.get_server(key)
        if original_server == removed_server:
            affected_keys.append(key)
    
    # Remove server and see reassignments
    hash_ring.remove_server(removed_server)
    reassigned_keys = {}
    
    for key in affected_keys:
        new_server = hash_ring.get_server(key)
        reassigned_keys[key] = new_server
    
    # New distribution after removal
    after_removal_distribution = hash_ring.analyze_distribution(test_keys)
    
    return {
        'initial_distribution': initial_distribution,
        'replicas': replicas,
        'affected_keys_count': len(affected_keys),
        'sample_reassignments': dict(list(reassigned_keys.items())[:5]),
        'after_removal_distribution': after_removal_distribution
    }

# Test consistent hashing
test_results = test_consistent_hashing()

print("Consistent Hashing Results:")
print(f"Initial server count: {test_results['initial_distribution']['server_count']}")
print(f"Total test keys: {test_results['initial_distribution']['total_keys']}")

print(f"\nInitial Load Distribution:")
for server, load in test_results['initial_distribution']['server_loads'].items():
    percentage = (load / test_results['initial_distribution']['total_keys']) * 100
    print(f"  {server}: {load:4d} keys ({percentage:5.1f}%)")

print(f"\nLoad Balance Metrics:")
print(f"Expected per server: {test_results['initial_distribution']['expected_per_server']:.0f}")
print(f"Min load: {test_results['initial_distribution']['min_load']}")
print(f"Max load: {test_results['initial_distribution']['max_load']}")
print(f"Balance ratio: {test_results['initial_distribution']['load_balance_ratio']:.3f}")

print(f"\nReplication Example:")
print(f"Key: important_data_key")
print(f"Replicas: {test_results['replicas']}")

print(f"\nServer Removal Impact:")
print(f"Keys affected by server-003 removal: {test_results['affected_keys_count']}")
print(f"Sample reassignments: {list(test_results['sample_reassignments'].items())[:3]}")

print(f"\nAfter Removal Load Balance:")
print(f"Balance ratio: {test_results['after_removal_distribution']['load_balance_ratio']:.3f}")

initial = test_results['initial_distribution']
after = test_results['after_removal_distribution']
assert initial['server_count'] == 5 and sum(initial['server_loads'].values()) == initial['total_keys']
assert after['server_count'] == 4 and sum(after['server_loads'].values()) == after['total_keys']
assert len(test_results['replicas']) == 3 and len(set(test_results['replicas'])) == 3
assert test_results['affected_keys_count'] > 0
assert 'server-003' not in test_results['sample_reassignments'].values()

print("Consistent hashing implementation completed successfully!")

In [ ]:
# Performance benchmarking and hash quality analysis
import time
import hashlib
from collections import defaultdict

print("=== MMH3 Performance Benchmarking and Hash Quality ===")

def benchmark_hash_functions():
    """Compare MMH3 with other hash functions"""
    test_data = [f"benchmark_string_{i:08d}" for i in range(50000)]
    
    # Hash functions to compare
    hash_functions = {
        'mmh3_32': lambda x: mmh3.hash(x.encode('utf-8')),
        'mmh3_128': lambda x: mmh3.hash128(x.encode('utf-8')),
        'python_hash': lambda x: hash(x),
        'md5': lambda x: int(hashlib.md5(x.encode('utf-8')).hexdigest()[:8], 16),
        'sha256': lambda x: int(hashlib.sha256(x.encode('utf-8')).hexdigest()[:8], 16)
    }
    
    results = {}
    
    for name, hash_func in hash_functions.items():
        start_time = time.time()
        
        # Warm-up run
        for item in test_data[:1000]:
            hash_func(item)
        
        # Actual benchmark
        start_time = time.time()
        hash_values = []
        for item in test_data:
            hash_val = hash_func(item)
            hash_values.append(hash_val)
        
        end_time = time.time()
        
        # Performance metrics
        total_time = end_time - start_time
        hashes_per_second = len(test_data) / total_time
        
        results[name] = {
            'total_time': total_time,
            'hashes_per_second': hashes_per_second,
            'hash_values': hash_values
        }
    
    return results, test_data

def analyze_hash_quality(hash_values: List[int], name: str) -> Dict:
    """Analyze quality of hash distribution"""
    if not hash_values:
        return {}
    
    # Convert to positive integers for analysis
    positive_hashes = [abs(h) for h in hash_values]
    
    # Distribution analysis with different bucket sizes
    bucket_sizes = [100, 1000, 10000]
    distribution_analysis = {}
    
    for bucket_size in bucket_sizes:
        buckets = [h % bucket_size for h in positive_hashes]
        bucket_counts = defaultdict(int)
        
        for bucket in buckets:
            bucket_counts[bucket] += 1
        
        # Calculate uniformity metrics
        expected_per_bucket = len(hash_values) / bucket_size
        filled_buckets = len(bucket_counts)
        max_bucket = max(bucket_counts.values()) if bucket_counts else 0
        min_bucket = min(bucket_counts.values()) if bucket_counts else 0
        
        # Chi-square test for uniformity
        chi_square = sum((count - expected_per_bucket) ** 2 / expected_per_bucket 
                        for count in bucket_counts.values())
        
        distribution_analysis[bucket_size] = {
            'filled_buckets': filled_buckets,
            'empty_buckets': bucket_size - filled_buckets,
            'max_bucket': max_bucket,
            'min_bucket': min_bucket,
            'chi_square': chi_square,
            'uniformity_ratio': min_bucket / max_bucket if max_bucket > 0 else 0
        }
    
    # Collision analysis
    unique_hashes = len(set(hash_values))
    collision_rate = (len(hash_values) - unique_hashes) / len(hash_values)
    
    # Bit distribution analysis (for 32-bit hashes)
    if hash_values and isinstance(hash_values[0], int):
        bit_counts = [0] * 32
        for hash_val in positive_hashes[:10000]:  # Sample for efficiency
            for bit_pos in range(32):
                if hash_val & (1 << bit_pos):
                    bit_counts[bit_pos] += 1
        
        # Expected is 50% for each bit position
        expected_bit_count = min(len(hash_values), 10000) // 2
        bit_balance = [abs(count - expected_bit_count) / expected_bit_count 
                      for count in bit_counts]
        avg_bit_imbalance = sum(bit_balance) / len(bit_balance)
    else:
        avg_bit_imbalance = None
    
    return {
        'total_hashes': len(hash_values),
        'unique_hashes': unique_hashes,
        'collision_rate': collision_rate,
        'distribution_analysis': distribution_analysis,
        'avg_bit_imbalance': avg_bit_imbalance
    }

# Run comprehensive benchmarks
print("Running hash function benchmarks...")
benchmark_results, test_data = benchmark_hash_functions()

print("\nPerformance Comparison:")
print(f"{'Function':<12} {'Time (s)':<10} {'Hashes/sec':<15}")
print("-" * 40)

performance_ranking = []
for name, results in benchmark_results.items():
    print(f"{name:<12} {results['total_time']:<10.3f} {results['hashes_per_second']:<15,.0f}")
    performance_ranking.append((name, results['hashes_per_second']))

# Rank by performance
performance_ranking.sort(key=lambda x: x[1], reverse=True)
print(f"\nPerformance Ranking:")
for i, (name, hps) in enumerate(performance_ranking, 1):
    print(f"{i}. {name}: {hps:,.0f} hashes/sec")

# Hash quality analysis
print(f"\nHash Quality Analysis:")
quality_results = {}

for name, results in benchmark_results.items():
    if name in ['mmh3_32', 'python_hash']:  # Analyze integer hash functions
        quality = analyze_hash_quality(results['hash_values'], name)
        quality_results[name] = quality
        
        print(f"\n{name.upper()} Quality Metrics:")
        print(f"  Unique hashes: {quality['unique_hashes']:,}/{quality['total_hashes']:,}")
        print(f"  Collision rate: {quality['collision_rate']*100:.6f}%")
        
        if quality['avg_bit_imbalance']:
            print(f"  Bit balance: {quality['avg_bit_imbalance']*100:.2f}% avg deviation")
        
        # Show distribution for 1000 buckets
        if 1000 in quality['distribution_analysis']:
            dist = quality['distribution_analysis'][1000]
            print(f"  Distribution (1000 buckets):")
            print(f"    Filled: {dist['filled_buckets']}/1000")
            print(f"    Min-Max: {dist['min_bucket']}-{dist['max_bucket']}")
            print(f"    Uniformity: {dist['uniformity_ratio']:.3f}")

mmh3_quality = quality_results['mmh3_32']
assert mmh3_quality['total_hashes'] == len(test_data)
assert mmh3_quality['collision_rate'] < 0.001, mmh3_quality['collision_rate']
assert mmh3_quality['distribution_analysis'][100]['filled_buckets'] == 100

# Memory usage comparison
print(f"\nMemory Usage Comparison:")
import sys

sample_hashes = {name: results['hash_values'][:1000] for name, results in benchmark_results.items()}
for name, hashes in sample_hashes.items():
    memory_size = sys.getsizeof(hashes) + sum(sys.getsizeof(h) for h in hashes)
    print(f"{name:<12}: {memory_size:,} bytes for 1000 hashes")

print("Performance benchmarking and hash quality analysis completed successfully!")

In [ ]:
# Production optimization and advanced MMH3 usage patterns
import os
import threading
from concurrent.futures import ThreadPoolExecutor
import queue

print("=== MMH3 Production Optimization and Advanced Usage ===")

class HashingWorkloadManager:
    def __init__(self, num_workers: int = 4):
        self.num_workers = num_workers
        self.results_queue = queue.Queue()
        self.stats = {
            'total_processed': 0,
            'processing_time': 0,
            'thread_stats': {}
        }
    
    def parallel_hash_processing(self, data_chunks: List[List[str]]) -> Dict:
        """Process hash operations in parallel using multiple threads"""
        import threading
        import time
        
        def worker_hash_function(chunk_id: int, data_chunk: List[str]):
            """Worker function for parallel hashing"""
            thread_id = threading.current_thread().ident
            start_time = time.time()
            
            chunk_results = []
            for item in data_chunk:
                # Use 128-bit hash for better distribution in production
                hash128 = mmh3.hash128(item.encode('utf-8'))
                hash32 = mmh3.hash(item.encode('utf-8'))
                
                chunk_results.append({
                    'item': item,
                    'hash32': hash32,
                    'hash128': hash128,
                    'hash32_hex': f"0x{hash32 & 0xffffffff:08x}"
                })
            
            end_time = time.time()
            
            # Store results and stats
            result = {
                'chunk_id': chunk_id,
                'thread_id': thread_id,
                'processing_time': end_time - start_time,
                'items_processed': len(data_chunk),
                'results': chunk_results
            }
            
            self.results_queue.put(result)
            return result
        
        # Execute parallel processing
        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            futures = []
            for i, chunk in enumerate(data_chunks):
                future = executor.submit(worker_hash_function, i, chunk)
                futures.append(future)
            
            # Collect results
            all_results = []
            for future in futures:
                result = future.result()
                all_results.append(result)
        
        # Aggregate statistics
        total_time = sum(r['processing_time'] for r in all_results)
        total_items = sum(r['items_processed'] for r in all_results)
        
        return {
            'total_items': total_items,
            'total_time': total_time,
            'avg_time_per_item': total_time / total_items if total_items > 0 else 0,
            'throughput': total_items / (max(r['processing_time'] for r in all_results) if all_results else 1),
            'worker_results': all_results
        }

class ProductionHashTable:
    def __init__(self, initial_size: int = 10007, load_factor_threshold: float = 0.75):
        """Production-ready hash table using MMH3"""
        self.initial_size = initial_size
        self.size = initial_size
        self.load_factor_threshold = load_factor_threshold
        self.buckets = [[] for _ in range(self.size)]
        self.count = 0
        self.resize_count = 0
        
    def _hash(self, key: str) -> int:
        """Hash function using MMH3 with current table size"""
        return abs(mmh3.hash(key.encode('utf-8'))) % self.size
    
    def _resize(self):
        """Resize hash table when load factor exceeds threshold"""
        old_buckets = self.buckets
        old_size = self.size
        
        # Double the size and find next prime-like number
        self.size = self._next_prime_like(self.size * 2)
        self.buckets = [[] for _ in range(self.size)]
        self.count = 0
        self.resize_count += 1
        
        # Rehash all existing items
        for bucket in old_buckets:
            for key, value in bucket:
                self.insert(key, value)
    
    def _next_prime_like(self, n: int) -> int:
        """Find next odd number (approximates prime for hash table sizing)"""
        if n % 2 == 0:
            n += 1
        return n
    
    def insert(self, key: str, value: any):
        """Insert key-value pair"""
        # Check load factor and resize if needed
        if self.count >= self.size * self.load_factor_threshold:
            self._resize()
        
        hash_index = self._hash(key)
        bucket = self.buckets[hash_index]
        
        # Check if key already exists
        for i, (existing_key, existing_value) in enumerate(bucket):
            if existing_key == key:
                bucket[i] = (key, value)  # Update existing
                return
        
        # Add new key-value pair
        bucket.append((key, value))
        self.count += 1
    
    def get(self, key: str) -> any:
        """Retrieve value by key"""
        hash_index = self._hash(key)
        bucket = self.buckets[hash_index]
        
        for existing_key, value in bucket:
            if existing_key == key:
                return value
        
        raise KeyError(key)
    
    def get_statistics(self) -> Dict:
        """Get hash table performance statistics"""
        bucket_lengths = [len(bucket) for bucket in self.buckets]
        non_empty_buckets = sum(1 for length in bucket_lengths if length > 0)
        
        return {
            'size': self.size,
            'count': self.count,
            'load_factor': self.count / self.size,
            'resize_count': self.resize_count,
            'non_empty_buckets': non_empty_buckets,
            'empty_buckets': self.size - non_empty_buckets,
            'max_bucket_length': max(bucket_lengths) if bucket_lengths else 0,
            'avg_bucket_length': sum(bucket_lengths) / len(bucket_lengths),
            'collision_chains': sum(1 for length in bucket_lengths if length > 1)
        }

# Test production workload management
print("Testing parallel hash processing...")

# Create test data
test_data = [f"production_key_{i:08d}" for i in range(10000)]

# Split into chunks for parallel processing
chunk_size = 2500
data_chunks = [test_data[i:i + chunk_size] for i in range(0, len(test_data), chunk_size)]

workload_manager = HashingWorkloadManager(num_workers=4)
parallel_results = workload_manager.parallel_hash_processing(data_chunks)

print(f"Parallel Processing Results:")
print(f"Total items: {parallel_results['total_items']:,}")
print(f"Total processing time: {parallel_results['total_time']:.3f}s")
print(f"Average time per item: {parallel_results['avg_time_per_item']*1000:.3f}ms")
print(f"Throughput: {parallel_results['throughput']:,.0f} items/sec")
assert parallel_results['total_items'] == len(test_data)
# Hashing in worker threads gives the same values as hashing in the main thread
assert all(r['hash32'] == mmh3.hash(r['item'].encode('utf-8'))
           for w in parallel_results['worker_results'] for r in w['results'][:20])

# Test production hash table
print(f"\nTesting production hash table...")

hash_table = ProductionHashTable(initial_size=1009)

# Insert test data
start_time = time.time()
for i, key in enumerate(test_data[:5000]):
    hash_table.insert(key, f"value_{i}")

insert_time = time.time() - start_time

# Test retrieval performance
start_time = time.time()
retrieved_count = 0
for key in test_data[:1000]:
    try:
        value = hash_table.get(key)
        retrieved_count += 1
    except KeyError:
        pass

retrieval_time = time.time() - start_time

# Get statistics
table_stats = hash_table.get_statistics()

print(f"Hash Table Performance:")
print(f"Insertions: 5000 items in {insert_time:.3f}s ({5000/insert_time:.0f} ops/sec)")
print(f"Retrievals: {retrieved_count}/1000 in {retrieval_time:.3f}s ({1000/retrieval_time:.0f} ops/sec)")
print(f"Load factor: {table_stats['load_factor']:.3f}")
print(f"Resize operations: {table_stats['resize_count']}")
print(f"Max collision chain: {table_stats['max_bucket_length']}")
print(f"Collision chains: {table_stats['collision_chains']}")
assert retrieved_count == 1000 and table_stats['count'] == 5000
assert table_stats['load_factor'] <= hash_table.load_factor_threshold

# Memory efficiency analysis
import sys
memory_usage = {
    'buckets': sys.getsizeof(hash_table.buckets),
    'total_items': sum(sys.getsizeof(bucket) for bucket in hash_table.buckets),
    'estimated_total': sys.getsizeof(hash_table.buckets) + sum(sys.getsizeof(bucket) for bucket in hash_table.buckets)
}

print(f"\nMemory Usage Analysis:")
print(f"Bucket structure: {memory_usage['buckets']:,} bytes")
print(f"Stored items: {memory_usage['total_items']:,} bytes")
print(f"Estimated total: {memory_usage['estimated_total']:,} bytes")
print(f"Memory per item: {memory_usage['estimated_total'] / table_stats['count']:.1f} bytes")

print("Production optimization and advanced usage patterns completed successfully!")